In [ ]:
!pip install firebase
## needed
!pip install langdetect

In [ ]:
#[1]
## All the imports are here ##
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output , HTML
from IPython.display import display
from firebase import firebase
import json
import matplotlib.pyplot as plt
from datetime import datetime
import os
from collections import Counter
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from langdetect import detect
import calendar
import nltk
from nltk.chat.util import Chat, reflections
# Download necessary NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [ ]:
#[2]
#add styles to the elemetes (use exampleWidgit.addClass("classNamee") and use the class name to style in css)
def loadStyles():
    # Display CSS
  display(HTML('''<style>
    .widget-box {
      display: flex;
      background-color: #0D0C27;
      border: 3px dashed #00bfff;
      padding: 20px;
      border-radius: 30px;
      color: #FFB23E;
      height:550px;

      width:auto;
    }
    .widget-label {
      color: #FFB23E;
    }

    .response-chat-box{
        color: #FFB23E;
    }


    .custom-upload {
      background-color: #0D0C27;
      border: 2px dashed #00bfff;
      padding: 20px;
      text-align: center;
      border-radius: 10px;
      color: #FFB23E;
    }


    .button {
      background-color: #ff9933;
      color: #000000;
      border: none;
      border-radius: 20px;
      padding: 20px;
      cursor: pointer;
      margin: 10px;
      width:  322px;
      height: 200px;
      font-size: 20px;
    }
    .button:hover {
      background-color: #ffcc66;
    }
 .back-button {
            background-color: transparent;
            color: white;
            border: 2px solid #debf37;
            border-radius: 20px;
            padding: 20px;
            margin: 10px;
            width: 300px;
            height: 200px;
            font-size: 20px;
            position: fixed;
            bottom: 10px;
            right: 10px;
        }
        .back-button:hover {
            background-color: rgba(0, 0, 255, 0.1); /* Light blue background on hover */
        }

        .back-button-inprogressdisplay{
          height: 50px;
            position:none;

        }

        .save-button {
        position: fixed;
        bottom: 10px;
        Left: 10px;
        }

        .delete-last-entry-button {
        position: fixed;
        bottom: 10px;
        Left: 25%;
        background-color: red;
        }



    .upload-button {
      background-color: #a6a18b;
      color: white;
      border: dashed black;
      border-radius: 20px;
      cursor: pointer;
      margin-top: 10px;
      center: center;
      width: 300px;
      height: 50px;
    }
    .upload-button:hover {
      background-color: #ffcc66;
    }
    .show_graph_button{

    }

    .widget-label {
      color: #FFB23E !important;
    }
    .widget-checkbox .widget-label {
      color: #FFB23E !important;
    }
    .title {
      font-size: 36px;
      font-weight: bold;
      text-align: center;
      color: white;
    }
    .warning {
      font-size: 10px;
      font-weight: bold;
      color: red;
    }

    .response-chat-box {
        color: black; /* Set text color to black */
        background-color: #b5baff;
        padding: 10px;
        border-radius: 30px;
    }

    .response-chat-box table {
        border-collapse: collapse;
        width: 100%;
    }

    .response-chat-box th, .response-chat-box td {
        border: 1px solid #ddd;
        padding: 8px;
        text-align: left;
    }

    .response-chat-box th {
        background-color: #f2f2f2;
    }

    .response-chat-box tr:hover {
        background-color: transparent; /* Remove background color on hover */
    }

    .response-chat-box tr {
        background-color: white; /* Ensure consistent background color for rows */
    }

  </style>'''))



In [ ]:
#[3]
#connect to firebase
firebase_url = 'https://learning911-p-default-rtdb.europe-west1.firebasedatabase.app/'

FBconn = firebase.FirebaseApplication(firebase_url, None)
#get data from database
def fetch_json_data():
    result = FBconn.get(firebase_url, '/data')
    if result:
      last_key = list(result.keys())[-1] if result else None
      last_entry = result[last_key] if last_key else None #get the last database uploaded
    if last_entry is None:
        print("No data found in Firebase.")
        return {}
    else:
        return last_entry

# Fetch and delete the last entry from the database
def delete_last_entry(e):
    # Fetch data from the database
    result = FBconn.get(firebase_url, '/data')

    if result:
        last_key = list(result.keys())[-1]  # Get the key of the last entry
        last_entry = result[last_key]  # Get the last entry
        print(f"Last entry to be deleted: {last_entry}")

        # Delete the last entry
        FBconn.delete(firebase_url, f'/data/{last_key}')
        print(f"Deleted entry with key: {last_key}")
    else:
        print("No data found in Firebase.")



In [ ]:
#[4]
def update_items(category_dropdown, item_dropdown, df):
    selected_category = category_dropdown.value
    filtered_items = df[df['Category'] == selected_category]['Item']
    item_dropdown.options = filtered_items.unique()

In [ ]:
#fetching the data of the operation per hour
def getOperationPerHourData(df):
    if 'Time' not in df.columns:
        raise ValueError("'Time' column is missing in the data")

    df['Time'] = pd.to_datetime(df['Time'], errors='coerce')
    plot_data = {}

    for user in df['User'].unique():
        user_df = df[df['User'] == user]
        user_df = user_df.sort_values(by='Time')

        start_time = None
        total_time = pd.Timedelta(0)
        operation_count = 0

        for _, row in user_df.iterrows():
            operation_count += 1

            if row['Description'] == "Open document":
                start_time = row['Time']
            elif row['Description'] == "Close document" and start_time is not None:
                total_time += row['Time'] - start_time
                start_time = None

        total_hours = total_time.total_seconds() / 3600

        if total_hours > 0:
            plot_data[user] = {
                'operations_per_hour': operation_count / total_hours,
                'total_operations': operation_count,
                'total_hours': total_hours
            }

    return pd.DataFrame.from_dict(plot_data, orient='index').reset_index()

In [ ]:
#showing the graph
def plot_graph(plot_df):
    if len(plot_df.columns) == 4:
        plot_df.columns = ['User', 'Operations per Hour', 'Total Operations', 'Total Hours']
    else:
        print("Error: Plot DataFrame does not have the expected number of columns.")
        return

    colors = plt.cm.get_cmap('tab10', len(plot_df['User']))

    plt.figure(figsize=(10, 6))
    bars = plt.bar(plot_df['User'], plot_df['Operations per Hour'], color=[colors(i) for i in range(len(plot_df))])

    plt.xlabel('Student')
    plt.ylabel('Operations per Hour')
    plt.title('Operations per Hour for Each Student')
    plt.xticks(rotation=45)
    plt.tight_layout()

    plt.show()

In [ ]:
#[5]
#hamza window: show operation per hour graph
def graph1(event):
    global filtered_df
    if filtered_df.empty:
        filtered_df = df

    plot_df = getOperationPerHourData(filtered_df)
    plot_graph(plot_df)


In [ ]:
#[6]
def open_display4(event):
  title = widgets.HTML('<h1 class="title">Operations per Hour graph</h1>')
  show_graph_button = newButton('Show Graph','button')
  show_graph_button.on_click(graph1)
  show_graph_button.add_class('show_graph_button')
  back_button =getBackToMainMenuButton()
  myDisplay('V',title,show_graph_button,back_button)

In [ ]:
#[7]
#Basel window
#project progress graph
#the graph gives an how the project has progress in a time period
def getDateAndActions():
  #array for dates ,array for number of actions done in a single day
  global dateArray
  global actionPerDate


  #use filter data if filtered done before else use all the data of the json file
  #### DONT DELETE THIS IF STATEMENT ####
  if filtered_df.empty:
    filteredData = df
  else:
    filteredData = filtered_df


  localdateArray=[]
  localactionPerDate=[]
  if(len(localdateArray)>0 or len(localactionPerDate)>0):
    localactionPerDate.clear()
    localdateArray.clear()
  if 'Time' not in filteredData.columns:
    print("error, not time column found")
  #convert Time column to datetime to extract date easily
  filteredData['Time'] = pd.to_datetime(filteredData['Time'])
  #iterate over each row
  for index,row in filteredData.iterrows():
    #get the date only
    Date = row['Time'].date()
    # if date is fouund add 1 to localactionPerDate in the respective place
    if Date in localdateArray:
      i=localdateArray.index(Date)
      localactionPerDate[i]=localactionPerDate[i]+1
    #if date not found then add it to dateArray and add 1 (first action) in actionPerDate
    else:
      localdateArray.append(Date)
      localactionPerDate.append(1)
  dateArray=localdateArray
  actionPerDate=localactionPerDate


#get months only from date
def extract_months(dateArray):
  months = []
  for date in dateArray:
    flag=0
    month = date.month
    #if month not in months add it
    for i in range(len(months)):
      if(months[i]==month):
        flag=1
    #month not found add month
    if(flag==0):
      months.append(month)
  return months

#get years only from date
def extract_years(dateArray):
  years = []
  for date in dateArray:
    flag=0
    year = date.year
    #if year not in years add it
    for i in range(len(years)):
      if(years[i]==year):
        flag=1
    #year not found add year
    if(flag==0):
      years.append(year)
  return years

#get days for specfic month
def getDaysInMonth(month):
    #get the number of days in the specified month
    days_in_month = calendar.monthrange(2000, int(month))[1]
    #create a list of days from 1 to the number of days in the month
    days = list(range(1, days_in_month + 1))
    return days

# Function to update days dropdown based on selected month and year
def update_starting_days(change=None):
    if change['new'] is not None:
        selected_month = int(change['new'])
        days = getDaysInMonth(selected_month)
        StartingDay.options = [None] + days
    else:
        StartingDay.options=[None]

# Function to update days dropdown based on selected month and year
def update_finishing_days(change=None):
    if change['new'] is not None:
        selected_month = int(change['new'])
        days = getDaysInMonth(selected_month)
        FinishingDay.options = [None] + days
    else:
        FinishingDay.options=[None]

# Function to build date range and cumulative actions array
def build_date_range(start_date, end_date, date_array, action_array):
    date_range = pd.date_range(start=start_date, end=end_date)
    cumulative_actions = []
    sum_actions = 0
    for date in date_range:
        if date.date() in date_array:
            index = date_array.index(date.date())
            sum_actions += action_array[index]
        cumulative_actions.append(sum_actions)
    return date_range, cumulative_actions

# Function to update graph based on selected starting and finishing dates
def update_graph(change=None):
    plt.clf
    if StartingYear.value is not None and StartingMonth.value is not None and StartingDay.value is not None and \
       FinishingYear.value is not None and FinishingMonth.value is not None and FinishingDay.value is not None:
        start_date = pd.Timestamp(year=int(StartingYear.value), month=int(StartingMonth.value), day=int(StartingDay.value))
        end_date = pd.Timestamp(year=int(FinishingYear.value), month=int(FinishingMonth.value), day=int(FinishingDay.value))
        if(start_date>=end_date):
          print("The starting date must be before the finishing date")
          return
        date_range, cumulative_actions = build_date_range(start_date, end_date, dateArray, actionPerDate)
        print("The total amount of actions done in this period of time is ",cumulative_actions[len(cumulative_actions)-1])
        plt.figure(figsize=(10, 6))
        plt.plot(date_range, cumulative_actions, marker='o', linestyle='-', color='b')
        plt.title('Project Progress Over Time')
        plt.xlabel('Date')
        plt.ylabel('Cumulative Actions')
        plt.grid(True)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print("Please select starting date and finishing date")


def on_operations_per_student_click(b):
    graph1()


def on_showGraph_click(b):
  update_graph()

In [ ]:
import ipywidgets as widgets
#build a draop down for (year, month, day)
def create_date_dropdowns(label):
    Month = widgets.Dropdown(
        options=[None] + list(extract_months(dateArray)),
        description=f'{label} Month:',
        disabled=False
    )
    Year = widgets.Dropdown(
        options=[None] + list(extract_years(dateArray)),
        description=f'{label} Year:',
        disabled=False
    )
    Day = widgets.Dropdown(
        options=[None],
        description=f'{label} Day:',
        disabled=False
    )

    #update the days drop down menw based on the month
    def update_days(change):
        if change['new'] is not None:
            Day.options = [None] + getDaysInMonth(change['new'])
        else:
            Day.options = [None]

    Month.observe(update_days, names='value')
    return Day, Month, Year

def Show_Project_progress_Form():
    global StartingDay, StartingMonth, StartingYear, FinishingDay, FinishingMonth, FinishingYear

    getDateAndActions()

    # Create dropdowns for starting and finishing dates
    StartingDay, StartingMonth, StartingYear = create_date_dropdowns("Starting")
    FinishingDay, FinishingMonth, FinishingYear = create_date_dropdowns("Finishing")

    # Button to show graph
    showGraph = newButton('Show Graph', 'button')
    showGraph.on_click(on_showGraph_click)

    # UI components
    chooseStartDate = widgets.HTML(value="<b style='color: #ffffff;'>Choose starting date</b>")
    staetDate = widgets.HBox([StartingYear, StartingMonth, StartingDay])

    chooseFinishDate = widgets.HTML(value="<b style='color: #ffffff;'>Choose finishing date</b>")
    finishDate = widgets.HBox([FinishingYear, FinishingMonth, FinishingDay])

    # Back button
    back_button = getBackToMainMenuButton()
    back_button.add_class('back-button-inprogressdisplay')

    # Display the form
    myDisplay('V', chooseStartDate, staetDate, chooseFinishDate, finishDate, showGraph, back_button)


In [ ]:
#Basel window
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def getNumberOfOperationsForAStudent(event):


    #use filtered data if filter done before else use all the data of the json file
    #### DONT DELETE THIS IF STATEMENT ####
    if filtered_df.empty:
      filteredData = df
    else:
      filteredData = filtered_df


    #extract all the students from the json file
    students = filteredData['User'].unique()

    #define all the working shifts
    morning_start = pd.to_datetime('08:00:00').time()
    morning_end = pd.to_datetime('15:59:59').time()
    afternoon_start = pd.to_datetime('16:00:00').time()
    afternoon_end = pd.to_datetime('23:59:59').time()
    night_start = pd.to_datetime('00:00:00').time()
    night_end = pd.to_datetime('07:59:59').time()

    #initialize the workShifts matrix
    #in the first elemnt in workShifts saves the name of the student
    workShifts = [[student, 0, 0, 0] for student in students]
    #iterate over each student
    for i in range(len(students)):
        #for each student, iterate over the operations
        for index, row in filteredData.iterrows():
            #iterate over the operations done by only student[i]
            if row['User'] == students[i]:
                #convert row['Time'] to dateTime
                operation_time = pd.to_datetime(row['Time']).time()
                #check if the operation is in the morning shift
                if morning_start <= operation_time <= morning_end:
                    #in the second elemnt in workShifts saves the number of operations done in morning shift
                    workShifts[i][1] += 1
                #check if the operation is in the afternoon shift
                elif afternoon_start <= operation_time <= afternoon_end:
                    #in the third elemnt in workShifts saves the number of operations done in afternoon shift
                    workShifts[i][2] += 1
                #check if the operation is in the night shift
                elif night_start <= operation_time <= night_end:
                    #in the fourth elemnt in workShifts saves the number of operations done in night shift
                    workShifts[i][3] += 1
    return workShifts


def plot_shift_operations(event):
    # Extract shift names and data for plotting
    shift_names = ['Morning', 'Afternoon', 'Night']

    # Get the number of operations for each student
    workShifts = getNumberOfOperationsForAStudent('event')

    # Create lists to hold shift data
    morning_counts = []
    afternoon_counts = []
    night_counts = []
    students = []

    # Populate the lists with the data
    for student_data in workShifts:
        student = student_data[0]
        students.append(student)
        morning_counts.append(student_data[1])
        afternoon_counts.append(student_data[2])
        night_counts.append(student_data[3])

    # Set up the bar width and positions
    bar_width = 0.2
    index = np.arange(len(students))

    # Create the plot
    fig, ax = plt.subplots(figsize=(12, 8))

    # Plot each shift with a different color
    bars1 = ax.bar(index - bar_width, morning_counts, bar_width, label='Morning', color='skyblue')
    bars2 = ax.bar(index, afternoon_counts, bar_width, label='Afternoon', color='lightgreen')
    bars3 = ax.bar(index + bar_width, night_counts, bar_width, label='Night', color='salmon')

    # Add labels and title
    ax.set_xlabel('Students')
    ax.set_ylabel('Number of Operations')
    ax.set_title('Number of Operations per Shift for Each Student')
    ax.set_xticks(index)
    ax.set_xticklabels(students, rotation=45, ha='right')
    ax.legend()

    # Display the plot
    plt.tight_layout()
    plt.show()

In [ ]:
#Basel window
#display to show working hours graph
def open_display7(event):
  title = widgets.HTML('<h1 class="title">Disterbution working hours graph</h1>')
  showGraphBtn = newButton('Show Graph','button')
  showGraphBtn.on_click(plot_shift_operations)
  showGraphBtn.add_class('showGraphBtn')
  back_button =getBackToMainMenuButton()
  myDisplay('V',title,showGraphBtn,back_button)

In [ ]:
#[9]
def newButton(description,className):
  b=widgets.Button(description=description)
  b.add_class(className)
  return b

In [ ]:
#[10]
def create_upload_button(fileType,multiple=False):
  upload_button = widgets.FileUpload(
    accept=fileType,
    multiple=multiple,  # Do not allow multiple files
    description='Browse JSON file to upload',
    layout=widgets.Layout(width='300px')
    )
  return upload_button

In [ ]:
#[11]
global mainDisplayV
mainDisplayV = widgets.VBox()
global mainDisplayH
mainDisplayH = widgets.HBox()

def myDisplay(layout,*args):
    global mainDisplay
    if layout == 'H':
      mainDisplayH.children=args
      mainDisplayV.children=[]
      mainDisplay = mainDisplayH
    elif layout == 'V':
      mainDisplayV.children=args
      mainDisplayH.children=[]
      mainDisplay = mainDisplayV
    else:

        mainDisplay = widgets.VBox(args)
    clear_output()
    loadStyles()
    display(mainDisplay)

In [ ]:
#[12]
#display to show progress graph of the filtired students
def open_display3(event):
  Show_Project_progress_Form()

In [ ]:
#[13]
#make global varible for filtered data (type dataframe)
def save_filter_json( document_name, user_name,event):
    global filtered_df
    # Filter based on the 'Document' and 'User' columns
    if document_name is None and user_name is None:
        filtered_df = df
    elif document_name is not None and user_name is None:
      filtered_df  = df[ (df['Document'] == document_name)]
    elif document_name is None and user_name is not None:
      filtered_df  = df[df['User'] == user_name]
    elif document_name is not None and user_name is not None:
      filtered_df  = df[(df['Document'] == document_name) & (df['User'] == user_name)]

In [ ]:
#[14]
#extract csv file for the filtered data
def create_excel(event):
        filtered_df.to_csv('filtered_data.csv', index=False)
        print("Filtered data saved successfully as 'filtered_data.csv'.")

In [ ]:
#[15]
#get user names from the last data uploaded to the database
def getUserNames():
  userNames = set(df['User'].dropna().unique()) #extract the user names
  return list(userNames)

In [ ]:
#[16]
#get project names from the last data uploaded to the database
def getProjectsNames():
  ProjectNames = set(df['Document'].dropna().unique()) #extract the project names
  return list(ProjectNames)

In [ ]:
#[17]
#filter display
def open_display2(event):
  title = widgets.HTML('<h1 class="title">Data Filtiring</h1>')
  warning=widgets.HTML('<h6 class="warning">TO display project progress graph please choose user= none</h6>')
  # Dropdown widgets
  #project list drop down
  project_dropdown = widgets.Dropdown(
      options=[None] +getProjectsNames() ,
      description='ProjectName:',
      disabled=False,
      layout=widgets.Layout(width='300px'),
      style={'description_width': 'initial'}
  )
  project_dropdown.add_class('widget-label')
  #user list drop down
  user_dropdown = widgets.Dropdown(
      options=[None]+ getUserNames(),
      description='User:',
      disabled=False,
      layout=widgets.Layout(width='300px'),
      style={'description_width': 'initial'}
  )
  user_dropdown.add_class('widget-label')

  save_button = widgets.Button(description='Save Filtered Data')
  save_button.on_click(lambda event: save_filter_json(project_dropdown.value, user_dropdown.value,event))

  create_excel_button = widgets.Button(description='Create Excel')
  create_excel_button.on_click(create_excel)

  back_button =getBackToMainMenuButton()
  myDisplay('V',title,warning,project_dropdown,user_dropdown,save_button,create_excel_button,back_button)

In [ ]:
def open_display5(event):
    title = widgets.HTML('<h1 class="title">Create Index</h1>')
    title1 = widgets.HTML('<h3 class="title">The list of elements are:</h3>')
    back_button =getBackToMainMenuButton()
    # Get word frequencies from the DataFrame
    word_frequencies = count_word_frequencies()
    # Create a Textarea widget to display the word frequencies
    text_area = widgets.Textarea(
        value="\n".join(word_frequencies),
        description='Word Frequencies:',
        disabled=True,
        layout=widgets.Layout(width='600px', height='300px')
    )

    # Display the widgets
    myDisplay('V',title,title1,text_area,back_button)

In [ ]:
#[18]
#save the uploaded json file to project and upload to firebase
def onSaveButton(upload,event):
  global df
  # File upload
  if upload.value:
    # Get the uploaded file content
    uploaded_file = list(upload.value.values())[0]
    content = uploaded_file['content'].decode('utf-8')
    # Load JSON content
    uploaded_json = json.loads(content)
    df=pd.DataFrame(fetch_json_data())
  else:
    print("No file uploaded.")
  # File Save to Database

    print("Save to Firebase button clicked")
  if uploaded_json:
        FBconn = firebase.FirebaseApplication(firebase_url, None)
        FBconn.post('/data', uploaded_json)
        print("Saving to Firebase...")
        # Save the JSON content to a file (as an example)
        with open('uploaded_file.json', 'w') as f:
            json.dump(uploaded_json, f)
        print("File saved successfully!")
  else:
        print("No JSON data to save.")


In [ ]:
#[19]
#Upload JSON file display
def open_display1(event):
  # File upload widget
  title = widgets.HTML('<h1 class="title">Upload JSON file to database</h1>')
  upload = create_upload_button('.json')
  upload.add_class('upload-button')
  save_button = newButton('Save','button')
  delete_Last_button = newButton('Delete Last','button')
  delete_Last_button.on_click(delete_last_entry)
  delete_Last_button.add_class('delete-last-entry-button')
  save_button.add_class('save-button')
  save_button.on_click(lambda event: onSaveButton(upload, event))
  back_button =getBackToMainMenuButton()
  myDisplay('V',title,upload,save_button,delete_Last_button,back_button)

In [ ]:
#[20]
#backButon
def getBackToMainMenuButton():
  back_button =newButton('Back','button')
  back_button.add_class('back-button')
  back_button.on_click(show_menu)
  return back_button

In [ ]:
#[21]
#Main Menu - This is The first Page
def show_menu(event):
  global StartChat
  StartChat=False
  title = widgets.HTML('<h1 class="title">Badger App</h1>')
  button1=newButton('Upload','button')
  button2=newButton('filter','button')
  button3=newButton('Progress','button')
  button4=newButton('Operations per Hour'  ,'button')
  button5=newButton('Create index'  ,'button')
  button6=newButton('Chat Bot','button')
  button7=newButton('Working hours', 'button')
  button1.on_click(open_display1)
  button2.on_click(open_display2)
  button3.on_click(open_display3)
  button4.on_click(open_display4)
  button5.on_click(open_display5)
  button6.on_click(open_display6)
  button7.on_click(open_display7)

  myDisplay('v',title,button1,button2,button3,button4,button5,button6,button7)


In [ ]:
import pandas as pd
import json
from collections import Counter
import re
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import nltk

# Download NLTK data if you haven't already
nltk.download('punkt')

# Initialize the DataFrame variables
df = pd.DataFrame()  # Replace with your actual data
filtered_df = pd.DataFrame()

def count_word_frequencies():
    global df
    global filtered_df

    # Check if the data was filtered
    if filtered_df.empty:
        filtered_df = df

    # Initialize the stemmer
    stemmer = PorterStemmer()

    # Convert DataFrame to a single string
    text = " ".join(filtered_df.astype(str).apply(lambda row: ' '.join(row), axis=1))

    # Tokenize the text into words
    words = word_tokenize(text.lower())

    # Filter out non-alphabetic words
    words = [word for word in words if word.isalpha()]

    # Apply stemming to each word
    stemmed_words = [stemmer.stem(word) for word in words]

    # Count the frequency of each word
    word_counts = Counter(stemmed_words)

    # Convert to a list of tuples and sort by frequency
    word_list = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)

    return [f"{word}: {count}" for word, count in word_list]

# Example usage:
# Make sure to assign data to df and filtered_df before calling


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
#chatbot
#Basel reworked this : change the fetching of the df to here instead of declaring it in the last notbook's code
global df
df=pd.DataFrame(fetch_json_data())

#editing the fromat ofthe displayed data
def Bot_getOperationPerHour(df):
    # Get the operation per hour data
    result_df = getOperationPerHourData(df)

    # Convert the DataFrame to a formatted HTML string without extra styles
    formatted_output = result_df.to_html(index=False, header=True, border=0)

    # Add a header to make it clear in the bot response
    header = "<b>Here is the Operation Per Hour data:</b><br><br>"

    # Combine the header, custom styles, and the formatted DataFrame HTML string
    return header + formatted_output


# Create input and response widgets
input_box = widgets.Text(
    value='',
    placeholder='Type your message here...',
    description='You:',
    disabled=False
)
response_box = widgets.HTML(value="<b>ChatBot:</b> How can I assist you today?")

# Define patterns and responses
patterns = [
    (r'(?i)\bhi\b|\bhello\b|\bhey\b', ['Hello!', 'Hi there!', 'Welcome to the project management assistant.']),
    (r'(?i)how are you\??', ['I\'m functioning well, thank you!', 'I\'m operational and ready to assist with your project management.']),
    (r'(?i)what is your name\??', ['I\'m the Project Management Assistant.', 'You can call me the Assistant.']),
    (r'(?i)\bwhat data do you have\b|\bwhat is in the database\b|\bwhat kind of data\b|\bwhat data\b',
     ["I have information related to project management data, including task progress, deadlines, resource allocation, and more. How can I assist you with this data?"]),
    (r'(?i)\bgive me the latest data\b|\bshow me the most recent\b|\bmost recent entry\b|\bcurrent status\b',
     ["Here is the most recent entry in the database:", "<recent data entry placeholder>"]),
    (r'(?i)\bsummary\b|\bstatistics\b|\bsummarize the data\b|\bdata overview\b|\bshow me the statistics\b',
     ["Here is a summary of the data:", "<data summary placeholder>"]),


    (r'(?i)\boperation\b|\boperations\b|\bget operation per hour data\b',
     ["I'm preparing the data using the provided DataFrame.\n\n" + Bot_getOperationPerHour(df)]),



]

# Create the chatbot
chatbot = Chat(patterns, reflections)
# Fetch data from Firebase
data = fetch_json_data()

def open_display6(event):
    title = widgets.HTML('<h1 class="title">Chat Bot</h1>')
    input_box.on_submit(on_input_chatbox_submit)
    input_box.add_class('input-chat-box')
    response_box.add_class('response-chat-box')
    back_button =getBackToMainMenuButton()
    myDisplay('V',title,input_box,response_box,back_button)

def on_input_chatbox_submit(event):
    user_input = input_box.value.strip().lower()  # Normalize input to lowercase
    user_input = re.sub(r'[^\w\s]', '', user_input)  # Remove punctuation
    response = chatbot.respond(user_input)
    response_box.value = f"<b>You:</b> {user_input}<br><b style='color: #168900;'>ChatBot:</b> {response if response is not None else 'This type of question is not supported.'}"
    input_box.value = ''  # Clear input after submission


In [ ]:
#[Last]
#main app call #
global filtered_df
global uploaded_json
show_menu('event')

In [ ]:
#Don't add cells under this cell

# **#Don't add cells under this cell**

In [ ]:
print(Bot_getOperationPerHour(df))

<b>Here is the Operation Per Hour data:</b><br><br><table class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>index</th>
      <th>operations_per_hour</th>
      <th>total_operations</th>
      <th>total_hours</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>StudentA</td>
      <td>48.979592</td>
      <td>388</td>
      <td>7.921667</td>
    </tr>
    <tr>
      <td>StudentB</td>
      <td>215.413534</td>
      <td>382</td>
      <td>1.773333</td>
    </tr>
  </tbody>
</table>
